# Software design I

In [ ]:
# Rule of thumb - you should be able to describe a function in 6 words

In [1]:
def double(x):
    return 2 * x

print(double)

<function double at 0x7f74b00bf880>


In [2]:
twotimes = double

print(twotimes)

<function double at 0x7f74b00bf880>


In [4]:
print(double(17))
print(twotimes(12))
print(twotimes(4) == double(4))

34
24
True


In [ ]:
twotimes() # Execute function
twotimes # Use function as object
list(map(twotimes, [2, 3, 4]))

[4, 6, 8]

In [7]:
def some_function():
    print("Ran some function")

def wrapper(func_to_run): # Use function as object
    print("Ran wrapper...")
    func_to_run() # Execute function
    print("Finished wrapper...")

wrapper(some_function)

Ran wrapper...
Ran some function
Finished wrapper...


In [ ]:
def my_decorator(func_to_run): # my_decorator is a function that creates a function
    def wrapper():
        print('Wrapper started')
        func_to_run()
        print('Wrapper ended')
    return wrapper

f = my_decorator(some_function) # Create a function 'f' using my_decorator
f()

Wrapper started
Ran some function
Wrapper ended


In [10]:
@my_decorator
def hello():
    print('Hello')

hello()

Wrapper started
Hello
Wrapper ended


In [11]:
f()

Wrapper started
Ran some function
Wrapper ended


In [ ]:
# @python_app
# decorator imported from parsl
# gets your function (the one you prepend it to) and does something with it
# in this case, sets up all the parsl stuff and passes the function to workers

# Global variables

In [ ]:
def task():
    x = 10

x = 5
task() # Running task does not change the value of x
print(x)

# local namespace - from local scope i.e. inside functions
# enclosing namespace - 
# global namespace - variables defined in main loop (i.e. not in function) or explicitly created global variables
# built-in namespace - variables from all packages

# python first searches for a variable in its local namespace
# for main loop (i.e. not in a function) the main loop is the local namespace

10


In [ ]:
def task():
    global x # Using global x, now whenever we call task() we overwrite x in the main loop
    x = 10

x = 5
task() # Because we used 'global', task() now changes the value of x
print(x)

5


In [ ]:
# explicilty declared global variables can cause lots of problems with parsl/parallelization
# e.g. one thread can change a global variable and create unexpected results in a totally separate thread

In [ ]:
a = 3
def do_stuff(b):
    return b * a # b is in local function scope, do_stuff doesn't find a in function scope, looks at enclosing environment (i.e. main loop) and finds a there

do_stuff(6)

18

In [ ]:
# pure function
# - takes inputs, only returns it return
# - does not have a side effect

# - e.g. function takes an input, writes out a file, returns something
# - NOT a pure function because it has a side effect
#   - it writes out a file
#   - writing out a file changes the status of a machine

# - e.g. function that returns the current time
# - NOT a pure function because it depends on the system time

# try to isolate non-pure functions and not include them in paralleization

# things that change the status of the machine (i.e. writing out files) often have locks
# i.e. a task will lock up a file so that other tasks can't change it simultaneously
# this is a problem for parallization

In [21]:
from concurrent.futures import ProcessPoolExecutor
import time
import multiprocessing

def hello(i):
    print(i, 'Hello')
    print(i, 'world')

executor = ProcessPoolExecutor()
futures = [executor.submit(hello, i) for i in range(3)]

for future in futures:
    future.result()

# The print screen is a shared resource
# So each of the futures/workers fires off print statements to the same shared resource
# The order of what gets printed is unexpected, and depends on how quickly each worker works

0 Hello12
0   HelloHelloworld


21  world
world


In [ ]:
def hello(i, lock):
    with lock: # The lock ensures that only one thread can do what's within the 'with' block at a time, because the whole 'hello' function is essentialy part of the 'with' block it means only one thread can run the hello function at a time
        print(i, 'Hello')
        print(i, 'world')

lock = multiprocessing.Manager().Lock() # lock object
executor = ProcessPoolExecutor()
futures = [executor.submit(hello, i, lock) for i in range(3)]

for future in futures:
    future.result()

# use lock for as little time as possible
# e.g. if you need to write out a file, lock, write out file, then remove lock

1 Hello
1 world
0 Hello
0 world
2 Hello
2 world
